this file takes a csv file and saves it as a batched file with log probs calculated

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from detector import extract_features, EssayAIDetector
import pandas as pd
from get_logprobs import LMLogProbs
from get_logprobs import cross_model_disagreement
import numpy as np
import joblib
import nltk
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV
from collections import Counter
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.metrics import f1_score
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
import re

In [ ]:
lm_models = {
    "gpt2": LMLogProbs("gpt2-medium"),
    #"pythia": LMLogProbs("EleutherAI/pythia-410m"),
    #"distil": LMLogProbs("distilgpt2")
    #"neo": LMLogProbs("EleutherAI/gpt-neo-1.3B")
}

In [ ]:
import pandas as pd
import torch
import json
from tqdm import tqdm
from itertools import chain
import os

# -----------------------------
# Settings
# -----------------------------
BATCH_SIZE = 16       # Number of sentences per batch
MAX_TOKENS = 1024     # Max tokens per sentence
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
window = 4

for model_name, lm in lm_models.items():
    if lm.tokenizer.pad_token is None:
        lm.tokenizer.pad_token = lm.tokenizer.eos_token
    lm.model.to(DEVICE)
    lm.model.half()
    lm.model.eval()

# -----------------------------
# Load data
# -----------------------------
#data3 = pd.read_csv("../datasets/AI_Human.csv")
#text_dataset = data3[["text", "generated"]].copy()

# -----------------------------
# Helper functions
# -----------------------------

def get_batch_token_logprobs_and_tokens(lm, texts, max_length=MAX_TOKENS):
    enc = lm.tokenizer(
        texts,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=max_length
    ).to(DEVICE)

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]
    lm.model.eval()

    out = lm.model(input_ids=input_ids, attention_mask=attention_mask)
    log_probs = torch.nn.functional.log_softmax(out.logits, dim=-1)
    token_logps = log_probs.gather(
        2, input_ids[:, 1:].unsqueeze(-1)
    ).squeeze(-1)

    lengths = attention_mask.sum(dim=1) - 1

    results = []
    for i in range(len(texts)):
        L = lengths[i].item()
        toks = lm.tokenizer.convert_ids_to_tokens(input_ids[i, 1:L+1])
        lps = token_logps[i, :L].cpu().numpy()
        results.append((toks, lps))

    return results

import os

output_dir = os.path.expanduser("~/AI/ai_detector/training_data/")
os.makedirs(output_dir, exist_ok=True)


# Open JSONL files for each model
sentence_dict = {
    "gpt2": open(output_dir + "sentence_logprobs_gpt2_raid-v3.jsonl", "w"),
    #"pythia": open(output_dir + "sentence_logprobs_pythia_raid.jsonl", "w")
}
#chunk_dict = {
#    "gpt2": open(output_dir + "chunk_logprobs_gpt2_AI_Human.jsonl", "w"),
#    #"pythia": open(output_dir + "chunk_logprobs_pythia_AI_Human.jsonl", "w")
#}
with torch.no_grad():
    with torch.cuda.amp.autocast():
        count = 0
        for row in tqdm(text_dataset.itertuples(index=False), total=len(text_dataset)):
            count = count + 1
            essay = row.generation
            if row.model == "human":
                label = 0
            else:
                label = 1

            # Split essay into sentences
            sentences = split_sentences_max_words(essay)

            # Process in batches
        # ---------- SENTENCE RECORDS ----------

            for i in range(0, len(sentences), BATCH_SIZE):
                for model_name, lm in lm_models.items():

                    batch_sentences = sentences[i:i+BATCH_SIZE]

                    results = get_batch_token_logprobs_and_tokens(lm, batch_sentences)

                    for sent, (tokens, logps) in zip(batch_sentences, results):
                        if len(logps) == 0:
                            continue

                        record = {
                            "text": sent,
                            "essayNum":count,
                            "model": model_name,
                            "label": label,
                            "word_log_probs": logps.tolist(),
                            "mean": float(np.mean(logps)),
                            "std": float(np.std(logps))
                        }

                        sentence_dict[model_name].write(json.dumps(record) + "\n")


            # ---------- CHUNK RECORDS ----------
            # build chunks ONCE
            #if len(sentences) <= window:
            #    chunks = [" ".join(sentences)]
            #else:
            #    chunks = []
            #    for j in range(0, len(sentences) - window + 1, 1):
            #        chunks.append(" ".join(sentences[j:j+window]))
#
            #for i in range(0, len(chunks), BATCH_SIZE):
            #    batch_chunks = chunks[i:i+BATCH_SIZE]
#
            #    for model_name, lm in lm_models.items():
            #        results = get_batch_token_logprobs_and_tokens(lm, batch_chunks)
#
            #        for chunk_text, (tokens, logps) in zip(batch_chunks, results):
            #            if len(logps) == 0:
            #                continue
#
            #            record = {
            #                "text": chunk_text,
            #                "essayNum":count,
            #                "model": model_name,
            #                "label": label,
            #                "word_log_probs": logps.tolist(),
            #                "mean": float(np.mean(logps)),
            #                "std": float(np.std(logps))
            #            }
#
            #            chunk_dict[model_name].write(json.dumps(record) + "\n")

        # Close files
    for f in sentence_dict.values():
        f.close()